# Study the influence of parameters on energy.

As explained in the [official documentation](https://mendevi.readthedocs.io/latest/build/api/mendevi.models.trend.TrendNormlisator.html), this protocol models the influence of parameter $X_k$ on output $Y$, independently of the other contextual inputs. So we have $\tilde{Y}=\tilde{\phi}(X_k)$ and $\hat{Y}=a \tilde{Y}+b$. The model trys to minimise the erreor betwwen $\hat{Y}$ and the gound truth $Y = \phi(X_i)$.

He makes the strong assumption that the context parameter acts independently on the output as an affine denormalization function.

In [ ]:
import matplotlib.pyplot as plt

from mendevi.cst.plot import FIGSIZE
from mendevi.models.trend import TrendEncodeEnergy

## Model Training

In [ ]:
# model declaration

interest_label = "pixels"  # "quality" for the crf impact, "spatial_dct" for the texture...

model = TrendEncodeEnergy(
    interest_labels=[interest_label],
    weights=f"trend_{interest_label}",  # optional, for checkpoint (in ~/.cache/mendevi)
    niter=1000,  # number of training iterations, default (10000)
    learning_rate=1e-5,  # default learning rate to 1e-4 (Adam gradient decay)
)

In [ ]:
# model training

dataset = "scatter.db"  # the dataset name, automaticaly downloaded
val = 0.2  # keep 80% of data for training, 20% for validation

if not model.is_fit():
    model.fit(
        dataset,
        table="t_enc_encode",  # optional, help to find the good SQL request
        select=f"uniform(vid_hash) >= {val}",  # keep 80% of data for training
    )

In [ ]:
model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val}")

## Visualisation

In [ ]:
fig = plt.figure(layout="constrained", figsize=FIGSIZE)
model.fill_fig_phi(fig)
plt.legend()
plt.show()